In [ ]:
import os
from pathlib import Path
from getpass import getpass

from PIL import Image
from IPython.display import Image as IPythonImage, display
from sentence_transformers import SentenceTransformer

import weaviate
import weaviate.classes as wvc
from weaviate.classes.init import Auth
from weaviate.classes.query import MetadataQuery

In [ ]:
IMAGE_DIR = Path("./images")
TEST_IMAGE_DIR = Path("./test_images")

print("Images:", IMAGE_DIR.exists())
print("Test images:", TEST_IMAGE_DIR.exists())

In [ ]:
if "WEAVIATE_URL" not in os.environ:
    os.environ["WEAVIATE_URL"] = getpass("WEAVIATE_URL: ")

if "WEAVIATE_API_KEY" not in os.environ:
    os.environ["WEAVIATE_API_KEY"] = getpass("WEAVIATE_API_KEY: ")

In [ ]:
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.environ["WEAVIATE_URL"],
    auth_credentials=Auth.api_key(os.environ["WEAVIATE_API_KEY"]),
)

print("Client ready:", client.is_ready())

In [ ]:
clip_model = SentenceTransformer("clip-ViT-B-32")

In [ ]:
def get_image_paths(image_dir: Path) -> list[Path]:
    return [
        path
        for path in sorted(image_dir.iterdir())
        if path.suffix.lower() in {".jpg", ".jpeg", ".png"}
    ]


def embed_image(path: Path) -> list[float]:
    image = Image.open(path).convert("RGB")
    vector = clip_model.encode(
        image,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
    return vector.tolist()


def embed_text(text: str) -> list[float]:
    vector = clip_model.encode(
        text,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
    return vector.tolist()

In [ ]:
COLLECTION_NAME = "ClipImageCloud"

if client.collections.exists(COLLECTION_NAME):
    client.collections.delete(COLLECTION_NAME)

images = client.collections.create(
    name=COLLECTION_NAME,
    vector_config=[
        wvc.config.Configure.Vectors.self_provided(
            name="image_vector",
            vector_index_config=wvc.config.Configure.VectorIndex.hnsw(),
        )
    ],
    properties=[
        wvc.config.Property(
            name="filename",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="local_path",
            data_type=wvc.config.DataType.TEXT,
        ),
    ],
)

print("Created collection:", COLLECTION_NAME)

In [ ]:
image_paths = get_image_paths(IMAGE_DIR)

print("Images found:", len(image_paths))

inserted = []

for path in image_paths:
    vector = embed_image(path)

    uuid = images.data.insert(
        properties={
            "filename": path.name,
            "local_path": str(path),
        },
        vector={
            "image_vector": vector,
        },
    )

    inserted.append(uuid)
    print("Inserted:", path.name, uuid)

print("Done. Inserted:", len(inserted))

In [ ]:
response = images.query.fetch_objects(
    limit=10,
    return_properties=["filename", "local_path"],
    include_vector=True,
)

for obj in response.objects:
    print("UUID:", obj.uuid)
    print("Filename:", obj.properties["filename"])
    print("Vector length:", len(obj.vector["image_vector"]))
    print("-" * 80)

In [ ]:
query="open sea beach"

In [ ]:
query = "open sea beach"

query_vector = embed_text(query)

response = images.query.near_vector(
    near_vector=query_vector,
    target_vector="image_vector",
    limit=3,
    return_properties=["filename", "local_path"],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    filename = obj.properties["filename"]

    print("Distance:", obj.metadata.distance)
    print("Filename:", filename)

    display(IPythonImage(filename=str(IMAGE_DIR / filename), width=300))
    print("-" * 80)

In [ ]:
queries = [
    "a dog outside",
    "a guitar",
    "a beach with sea",
    "an office building",
    "a bird",
    "a tower",
]

for query in queries:
    print("\nQUERY:", query)

    query_vector = embed_text(query)

    response = images.query.near_vector(
        near_vector=query_vector,
        target_vector="image_vector",
        limit=3,
        return_properties=["filename", "local_path"],
        return_metadata=MetadataQuery(distance=True),
    )

    for obj in response.objects:
        filename = obj.properties["filename"]

        print("Distance:", obj.metadata.distance)
        print("Filename:", filename)

        display(IPythonImage(filename=str(IMAGE_DIR / filename), width=240))
        print("-" * 80)

In [ ]:
query_image = TEST_IMAGE_DIR / "Guitar_LIL_134218.jpg"

display(IPythonImage(filename=str(query_image), width=240))

In [ ]:
query_vector = embed_image(query_image)

response = images.query.near_vector(
    near_vector=query_vector,
    target_vector="image_vector",
    limit=3,
    return_properties=["filename", "local_path"],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    filename = obj.properties["filename"]

    print("Distance:", obj.metadata.distance)
    print("Filename:", filename)

    display(IPythonImage(filename=str(IMAGE_DIR / filename), width=300))
    print("-" * 80)

In [ ]:
response = images.query.fetch_objects(
    limit=50,
    return_properties=["filename", "local_path"],
)

for obj in response.objects:
    print(obj.uuid, obj.properties["filename"])

In [ ]:
client.close()